Modeling Features: Dataset Offsets
==================================

Multi-wavelength datasets often have offsets between their images, which are due to the different telescope pointings
during the observations.

These offsets are often accounted for during the data reduction process, which aligns the images, however:

 - Certain data reduction pipelines may not perfectly align the images, and the scientist may be unsure what the
 true offset between the images are.

 - Even if the reduction process does align the images, there is still a small uncertainty in the offset due to the
   precision of the telescope pointing which for detailed lens models must be accounted for.

This script shows how to include both an offset (two free parameters: y and x shifts) and a rotation (one free
parameter: roll angle in degrees) in the model for every additional dataset after the first. Together these
describe the rigid-body misalignment of each dataset relative to the reference (first) dataset, which is the
common pattern for multi-band space-telescope data (e.g. JWST NIRCam frames at slightly different roll angles).

To apply the misalignment, the code subtracts the offset from the grids aligned to the dataset pixels and then
rotates them about the offset point before performing lensing calculations. The light and mass model centres
do not change; only the coordinates of the image pixels input into these profiles are transformed.

When the source is reconstructed on a pixelization, these per-dataset offsets pair naturally with the shared
source-mesh feature (`shared_preloads=True`, see `features/same_wavelength/modeling.py`): the shared Delaunay
mesh is defined in the reference dataset's frame and every offset dataset's grid is traced onto it.

__Contents__

- **Advantages & Disadvantages:** If one fits a lens model to one dataset and applies it to other datasets, it is common to see the.
- **Model:** Compose the lens model fitted to the data.
- **Colors:** The colors of the multi-wavelength image, which in this case are green (g-band) and red (r-band).
- **Pixel Scales:** Every multi-wavelength dataset can have its own unique pixel-scale.
- **Dataset & Mask:** Standard set up of the dataset and mask that is fitted.
- **Analysis:** Create the Analysis object that defines how the model is fitted to the data.
- **Search:** Configure the non-linear search used to fit the model.
- **Result:** Overview of the results of the model-fit.

__Advantages__

If one fits a lens model to one dataset and applies it to other datasets, it is common to see the lens model fit
and source reconsturction degrade due to small offsets between the datasets. The same issue persists for simultaneous
fits to multiple datasets, even when care has been taken to align the datasets.

The advantage is therefore simple, for most multi-wavelength lens modeling, accounting for offsets in this way
is the only way to ensure the lens model is accurate and the source reconstruction is reliable.

__Disadvantages__

Each offset introduces two additional free parameters into the model for each dataset after the first dataset. For
4 datasets, this is 6 additional free parameters. This increases the dimensionality of the non-linear parameter space
and therefore the computational run-time of the model-fit.

__Model__

This script fits an `Imaging` dataset of a 'galaxy-scale' strong lens with a model where:

 - The lens galaxy's light is a an MGE bulge.
 - The lens galaxy's total mass distribution is an `Isothermal`; the external shear is an
   `ExternalShear` held in a `MassField`.
 - The source galaxy's light is a an MGE.

__Start Here Notebook__

If any code in this script is unclear, refer to the `multi_dataset/start_here.ipynb` notebook.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/autolens_workspace/blob/main/start_here.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Colors__

The colors of the multi-wavelength image, which in this case are green (g-band) and red (r-band).

The strings are used for load each dataset.

In [ ]:
waveband_list = ["g", "r"]

__Pixel Scales__

Every multi-wavelength dataset can have its own unique pixel-scale.

In [ ]:
pixel_scales_list = [0.08, 0.12]

__Dataset__

Load and plot each multi-wavelength strong lens dataset, using a list of their waveband colors.

The plotted images show that the datasets have a small offset between them, half a pixel based on the resolution of
the second image.

In [ ]:
dataset_type = "multi_dataset"
dataset_label = "imaging"
dataset_name = "dataset_offsets"

dataset_path = Path("dataset") / dataset_type / dataset_label / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/multi_dataset/features/dataset_offsets/simulator.py"],
        check=True,
    )

dataset_list = [
    al.Imaging.from_fits(
        data_path=Path(dataset_path) / f"{waveband}_data.fits",
        psf_path=Path(dataset_path) / f"{waveband}_psf.fits",
        noise_map_path=Path(dataset_path) / f"{waveband}_noise_map.fits",
        pixel_scales=pixel_scales,
    )
    for waveband, pixel_scales in zip(waveband_list, pixel_scales_list)
]

for dataset in dataset_list:
    aplt.subplot_imaging_dataset(dataset=dataset)

__Mask__

Define a 3.0" circular mask, which includes the emission of the lens and source galaxies.

For multi-wavelength lens modeling, we use the same mask for every dataset whenever possible. This is not
absolutely necessary, but provides a more reliable analysis.

The small offset between datasets means that the mask may not contain the exact same area of the image for every
dataset. 

For this dataset's offset of half a pixel (and anything of order a few pixels) this is fine and wont impact the analysis. 
However, for larger offsets the mask may need to be adjusted to ensure the same image area is masked out.

In [ ]:
mask_radius = 3.0

mask_list = [
    al.Mask2D.circular(
        shape_native=dataset.shape_native,
        pixel_scales=dataset.pixel_scales,
        radius=mask_radius,
    )
    for dataset in dataset_list
]

dataset_list = [
    dataset.apply_mask(mask=mask) for imaging, mask in zip(dataset_list, mask_list)
]

for dataset in dataset_list:
    aplt.subplot_imaging_dataset(dataset=dataset)

__Analysis__

We create an `Analysis` object for every dataset.

In [ ]:
analysis_list = [al.AnalysisImaging(dataset=dataset) for dataset in dataset_list]

__Model__

We compose a lens model where:

 - Parameters which shift (y_offset_0, x_offset_0) and rotate (grid_rotation_angle) the second dataset's image
 relative to the first dataset's image are included via the `DatasetModel` object [3 parameters].

 - The lens galaxy's light is an MGE with 1 x 20 Gaussians [6 parameters].

 - The lens galaxy's total mass distribution is an `Isothermal`; the external shear is an
   `ExternalShear` held in a `MassField` [7 parameters].

 - The source galaxy's light is an MGE with 1 x 20 Gaussians [4 parameters].

The number of free parameters and therefore the dimensionality of non-linear parameter space is N=24.

In [ ]:
bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=True,
    sigma_min=dataset_list[0].pixel_scales[0] / 10.0,
)

lens = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=bulge,
    mass=al.mp.Isothermal,
)

bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=False,
)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)

dataset_model = af.Model(al.DatasetModel)

field = af.Model(al.MassField, redshift=0.5, shear=af.Model(al.mp.ExternalShear))

model = af.Collection(
    dataset_model=dataset_model,
    galaxies=af.Collection(lens=lens, source=source),
    fields=field,
)

The `DatasetModel` can apply a shift to each dataset, however we do not want this to be applied to both
datasets as they will then both have free parameters for the same offset, duplicating free parameters.

The default prior on a `DatasetModel`'s offsets are actually not a prior, but fixed values of (0.0, 0.0),
meaning that if we do not update the model the shift will not be applied to the datasets.

We therefore update the `DatasetModel` below, to only apply a shift to the second dataset, which is the r-band image.

If we add more datasets, the code will apply the shift to each one after the first dataset, which are all shifted
relative to the first dataset, making it the reference point.

In [ ]:
analysis_factor_list = []

for i, analysis in enumerate(analysis_list):
    model_analysis = model.copy()

    if i > 0:
        model_analysis.dataset_model.grid_offset.grid_offset_0 = af.UniformPrior(
            lower_limit=-1.0, upper_limit=1.0
        )
        model_analysis.dataset_model.grid_offset.grid_offset_1 = af.UniformPrior(
            lower_limit=-1.0, upper_limit=1.0
        )
        # Roll-angle misalignment between exposures (degrees, CCW about the offset point).
        model_analysis.dataset_model.grid_rotation_angle = af.UniformPrior(
            lower_limit=-5.0, upper_limit=5.0
        )

    analysis_factor = af.AnalysisFactor(prior_model=model_analysis, analysis=analysis)

    analysis_factor_list.append(analysis_factor)

factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

__Search__

In [ ]:
search = af.Nautilus(
    path_prefix=Path("multi_dataset", "modeling"),
    name="dataset_offsets",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

__Model-Fit__

In [ ]:
result_list = search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)

__Result__

The result object returned by this model-fit is a list of `Result` objects, because we used a factor graph.
Each result corresponds to each analysis, and therefore corresponds to the model-fit at that wavelength.

In [ ]:
print(result_list[0].max_log_likelihood_instance)
print(result_list[1].max_log_likelihood_instance)

Plotting each result's tracer shows that the source appears different, owning to its different intensities.

In [ ]:
for result in result_list:
    aplt.subplot_tracer(tracer=result.max_log_likelihood_tracer, grid=result.grids.lp)

    aplt.subplot_fit_imaging(fit=result.max_log_likelihood_fit)

The `Samples` object still has the dimensions of the overall non-linear search (in this case N=15). 

Therefore, the samples is identical in every result object.

In [ ]:
for result in result_list:
    aplt.corner_anesthetic(samples=result.samples)

Checkout `autolens_workspace/*/guides/results` for a full description of analysing results.